In [ ]:
# Run this cell first in Colab
from google.colab import drive
drive.mount('/content/drive')

# Clone your repo
import os
repo_path = '/content/Brain_Tumor_Classification'

if not os.path.exists(repo_path):
    !git clone https://github.com/DevashreeNaidu/Brain_Tumor_Classification.git
    
os.chdir(repo_path)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip install -q tensorflow numpy matplotlib scikit-learn pillow seaborn

# Add src to path so we can import our modules
import sys
sys.path.append('/content/Brain_Tumor_Classification')

# Set data paths for Colab
import os
TRAIN_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Training'
TEST_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Testing'
MODELS_DIR = '/content/drive/MyDrive/MS/Project Tumor/models'
FIGURES_DIR = '/content/drive/MyDrive/MS/Project Tumor/results/figures'

# Create directories if they don't exist
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Paths configured")
print(f"Training data: {TRAIN_DIR}")
print(f"Models will save to: {MODELS_DIR}")

In [ ]:
import sys
sys.path.append('/content/Brain_Tumor_Classification')

from src.preprocessing import get_datasets

TRAIN_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Training'
TEST_DIR = '/content/drive/MyDrive/MS/Project Tumor/data/raw/Testing'

print("Creating datasets...")
train_ds, train_aug_ds, val_ds, test_ds, n_train, n_val = get_datasets(
    TRAIN_DIR, TEST_DIR, batch_size=32
)
print("Datasets ready.")

In [ ]:
import sys
sys.path.append('/content/Brain_Tumor_Classification')

from src.train import train_baseline, train_mobilenetv2, train_resnet50
from src.evaluate import evaluate_model, plot_training_history, summarize_all_experiments

results = []

# E1 — Baseline CNN, no augmentation
model_e1, history_e1 = train_baseline(train_ds, val_ds, augment=False)
plot_training_history(history_e1, 'E1 Baseline No Augmentation')
results.append(evaluate_model(model_e1, test_ds, 'E1 Baseline No Augmentation'))

# E2 — Baseline CNN + augmentation
model_e2, history_e2 = train_baseline(train_aug_ds, val_ds, augment=True)
plot_training_history(history_e2, 'E2 Baseline Augmentation')
results.append(evaluate_model(model_e2, test_ds, 'E2 Baseline Augmentation'))

# E3 — MobileNetV2
model_e3, history_e3_head, history_e3_ft = train_mobilenetv2(train_ds, val_ds)
plot_training_history(history_e3_head, 'E3 MobileNetV2 Phase 1')
plot_training_history(history_e3_ft, 'E3 MobileNetV2 Phase 2')
results.append(evaluate_model(model_e3, test_ds, 'E3 MobileNetV2'))

# E4 — ResNet50
model_e4, history_e4_head, history_e4_ft = train_resnet50(train_ds, val_ds)
plot_training_history(history_e4_head, 'E4 ResNet50 Phase 1')
plot_training_history(history_e4_ft, 'E4 ResNet50 Phase 2')
results.append(evaluate_model(model_e4, test_ds, 'E4 ResNet50'))

# Summary
summarize_all_experiments(results)